# Flex32 PSD difference notebook with switchable `e` / `x` / `v` parameterization

This notebook keeps the PSD glossy image as the conditioning input and trains on the difference target `glossy - diffuse`. It uses the separate `ParamDiffuser.py` module so the original repo files stay unchanged, and the sampling path is correct for `e`, `x`, and `v` prediction.

Set `TRAIN_TYPE` to `'e'`, `'x'`, or `'v'` in the config cell and rerun from the model cell downward.

In [ ]:
from __future__ import annotations

import math
import random
import sys
from collections import OrderedDict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / 'Run_Training.py').exists():
    candidates = [ROOT / 'FoilDIff', *ROOT.parents]
    for candidate in candidates:
        if (candidate / 'Run_Training.py').exists() and (candidate / 'Backbone.py').exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import Backbone
import ParamDiffuser as diff


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = ROOT / 'models' / '32'
CHECKPOINTS_DIR = ROOT / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo root: {ROOT}')
print(f'device: {DEVICE}')

In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 10
NOISE_STEPS = 200
EPOCHS = 2000
LR = 1e-4
FINAL_LR = 1e-5
WARMUP_EPOCHS = 100
EMA_DECAY = 0.9999

TRAIN_TYPE = 'e'  # switch between 'e', 'x', and 'v'
EVAL_EVERY = 200
CHECKPOINT_EVERY = 500
VAL_BATCHES = 4
VAL_CASE_INDEX = 0
EVAL_SEED = 123

PSD_ROOT = Path('/Users/27171653/Desktop/PhD/Highlight-modelling/PSD_Dataset/PSD_Dataset')
TRAIN_GLOSSY_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_specular'
TRAIN_DIFFUSE_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_diffuse'
VAL_GLOSSY_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_specular'
VAL_DIFFUSE_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_diffuse'

VALID_TRAIN_TYPES = {'e', 'x', 'v'}
if TRAIN_TYPE not in VALID_TRAIN_TYPES:
    raise ValueError(f'TRAIN_TYPE must be one of {sorted(VALID_TRAIN_TYPES)}, got {TRAIN_TYPE!r}')

SAVE_STEM = f'PSD_Difference_{TRAIN_TYPE}_Flex32'
TRAINER_SAVE_PATH = CHECKPOINTS_DIR / f'{SAVE_STEM}_checkpoint'
FINAL_MODEL_PATH = MODELS_DIR / f'{SAVE_STEM}.pth'

print(f'PSD root: {PSD_ROOT}')
print(f'train type: {TRAIN_TYPE}')
print(f'final model path: {FINAL_MODEL_PATH}')

In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert('RGB').resize((image_size, image_size), Image.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


class PairedPSDDifferenceDataset(Dataset):
    def __init__(self, glossy_dir: Path, diffuse_dir: Path, image_size: int = 32):
        self.glossy_dir = Path(glossy_dir)
        self.diffuse_dir = Path(diffuse_dir)
        self.image_size = image_size

        if not self.glossy_dir.exists():
            raise FileNotFoundError(f'Missing glossy directory: {self.glossy_dir}')
        if not self.diffuse_dir.exists():
            raise FileNotFoundError(f'Missing diffuse directory: {self.diffuse_dir}')

        glossy_files = {path.name: path for path in self.glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS}
        diffuse_files = {path.name: path for path in self.diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS}
        self.names = sorted(set(glossy_files) & set(diffuse_files))
        if not self.names:
            raise RuntimeError(f'No paired PSD samples found in {self.glossy_dir} and {self.diffuse_dir}')

        self.glossy_files = glossy_files
        self.diffuse_files = diffuse_files

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, idx: int):
        name = self.names[idx]
        glossy_path = self.glossy_files[name]
        diffuse_path = self.diffuse_files[name]

        glossy = load_rgb_tensor(glossy_path, self.image_size)
        diffuse = load_rgb_tensor(diffuse_path, self.image_size)
        difference = glossy - diffuse
        meta = {
            'name': name,
            'glossy_path': str(glossy_path),
            'diffuse_path': str(diffuse_path),
        }
        return glossy, difference, diffuse, meta


train_dataset = PairedPSDDifferenceDataset(TRAIN_GLOSSY_DIR, TRAIN_DIFFUSE_DIR, image_size=IMAGE_SIZE)
val_dataset = PairedPSDDifferenceDataset(VAL_GLOSSY_DIR, VAL_DIFFUSE_DIR, image_size=IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

fixed_val_condition, fixed_val_target, fixed_val_diffuse, fixed_val_meta = val_dataset[VAL_CASE_INDEX]
fixed_val_condition = fixed_val_condition.unsqueeze(0).to(DEVICE)
fixed_val_target = fixed_val_target.unsqueeze(0).to(DEVICE)
fixed_val_diffuse = fixed_val_diffuse.unsqueeze(0).to(DEVICE)

eval_generator = torch.Generator()
eval_generator.manual_seed(EVAL_SEED)
fixed_eval_noise = torch.randn(fixed_val_target.shape, generator=eval_generator, dtype=fixed_val_target.dtype).to(DEVICE)

print(f'train dataset size: {len(train_dataset)}')
print(f'val dataset size: {len(val_dataset)}')
print(f'train batches per epoch: {len(train_loader)}')
print(f'fixed val case: {fixed_val_meta["name"]}')
print(f'fixed condition shape: {tuple(fixed_val_condition.shape)}')
print(f'fixed difference target shape: {tuple(fixed_val_target.shape)}')
print(f'fixed diffuse shape: {tuple(fixed_val_diffuse.shape)}')

In [ ]:
def get_cosine_lambda(initial_lr: float, final_lr: float, epochs: int, warmup_epoch: int):
    def cosine_lambda(epoch_idx: int) -> float:
        if epoch_idx < warmup_epoch:
            return epoch_idx / max(warmup_epoch, 1)
        cosine = (math.cos((epoch_idx - warmup_epoch) / max(epochs - warmup_epoch, 1) * math.pi) + 1.0) / 2.0
        return 1.0 - (1.0 - cosine) * (1.0 - final_lr / initial_lr)

    return cosine_lambda


def checkpoint_save(model, optimizer, loss: float, epoch: int, save_path: Path, parameterization: str):
    model_dir = Path(f'{save_path}_epoch_{epoch}_loss_{loss:.4f}')
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': int(epoch),
            'loss': float(loss),
            'train_type': parameterization,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },
        model_dir / 'model.pth',
    )


def update_ema(ema_model, model, decay: float = 0.9999):
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())
    for name, param in model_params.items():
        if name in ema_params:
            ema_params[name].data.mul_(decay).add_(param.data, alpha=1.0 - decay)


model = Backbone.Flex(size=IMAGE_SIZE, noise_steps=NOISE_STEPS).to(DEVICE)
ema_model = deepcopy(model).to(DEVICE)
ema_model.load_state_dict(model.state_dict())
ema_model.eval()

diffuser = diff.CosSchDiffuser(steps=NOISE_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_cosine_lambda(initial_lr=LR, final_lr=FINAL_LR, epochs=EPOCHS, warmup_epoch=WARMUP_EPOCHS),
)

num_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'model: {model.__class__.__name__}')
print(f'trainable parameters: {num_params:,}')
print(f'optimizer: AdamW(lr={LR})')
print(f'diffuser: {diffuser.name}, steps={diffuser.steps}')

In [ ]:
def train_step(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device, train_type: str = 'e') -> torch.Tensor:
    condition, targets, _diffuse, _meta = batch
    condition = condition.to(device)
    targets = targets.to(device)

    batch_size = condition.shape[0]
    t = torch.randint(0, diffuser.steps, (batch_size,), dtype=torch.long, device=device)
    noise = torch.randn_like(targets)
    noisy_xt = diffuser.forward_diffusion(targets, t, noise)
    target = diffuser.training_target(targets, t, noise, parameterization=train_type)
    prediction = model(noisy_xt, t, condition)
    return F.mse_loss(prediction, target)


def evaluate_noise_loss(model: torch.nn.Module, loader, diffuser: diff.Diffuser, device: torch.device, train_type: str = 'e', max_batches: int | None = None) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for idx, batch in enumerate(loader):
            if max_batches is not None and idx >= max_batches:
                break
            losses.append(float(train_step(model, batch, diffuser, device, train_type=train_type).item()))
    return float(np.mean(losses)) if losses else float('nan')


def channel_limits(target_channel: torch.Tensor, prediction_channel: torch.Tensor):
    low = min(float(target_channel.min()), float(prediction_channel.min()))
    high = max(float(target_channel.max()), float(prediction_channel.max()))
    if abs(high - low) < 1e-8:
        high = low + 1e-8
    return low, high


def normalize_difference_rgb(target_rgb: torch.Tensor, prediction_rgb: torch.Tensor):
    pair = torch.stack([target_rgb, prediction_rgb], dim=0)
    low = pair.amin(dim=(0, 2, 3), keepdim=True)
    high = pair.amax(dim=(0, 2, 3), keepdim=True)
    scale = (high - low).clamp_min(1e-6)
    target_norm = ((target_rgb.unsqueeze(0) - low) / scale).squeeze(0).clamp(0.0, 1.0)
    prediction_norm = ((prediction_rgb.unsqueeze(0) - low) / scale).squeeze(0).clamp(0.0, 1.0)
    return target_norm, prediction_norm


def plot_prediction(target_tensor: torch.Tensor, prediction_tensor: torch.Tensor, condition_tensor: torch.Tensor, true_diffuse_tensor: torch.Tensor, epoch: int, train_type: str):
    target = target_tensor.detach().cpu()
    prediction = prediction_tensor.detach().cpu()
    condition = condition_tensor.detach().cpu()
    true_diffuse = true_diffuse_tensor.detach().cpu()
    predicted_diffuse = (condition - prediction).clamp(0.0, 1.0)
    diffuse_error = (predicted_diffuse - true_diffuse).abs().mean(dim=0)
    rgb_target, rgb_prediction = normalize_difference_rgb(target, prediction)
    rgb_error = (target - prediction).abs().mean(dim=0)

    fig, axes = plt.subplots(5, 3, figsize=(12, 18))
    channel_names = ['Red', 'Green', 'Blue']

    for channel_idx, channel_name in enumerate(channel_names):
        target_channel = target[channel_idx]
        prediction_channel = prediction[channel_idx]
        error_channel = (target_channel - prediction_channel).abs()
        vmin, vmax = channel_limits(target_channel, prediction_channel)

        axes[channel_idx, 0].imshow(target_channel, cmap='coolwarm', vmin=vmin, vmax=vmax)
        axes[channel_idx, 0].set_title(f'{channel_name} target')
        axes[channel_idx, 1].imshow(prediction_channel, cmap='coolwarm', vmin=vmin, vmax=vmax)
        axes[channel_idx, 1].set_title(f'{channel_name} prediction')
        axes[channel_idx, 2].imshow(error_channel, cmap='magma')
        axes[channel_idx, 2].set_title(f'{channel_name} abs error')

    axes[3, 0].imshow(rgb_target.permute(1, 2, 0).numpy())
    axes[3, 0].set_title('RGB difference target')
    axes[3, 1].imshow(rgb_prediction.permute(1, 2, 0).numpy())
    axes[3, 1].set_title('RGB difference prediction')
    axes[3, 2].imshow(rgb_error.numpy(), cmap='magma')
    axes[3, 2].set_title('RGB mean abs error')

    axes[4, 0].imshow(true_diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 0].set_title('True reconstructed diffuse')
    axes[4, 1].imshow(predicted_diffuse.permute(1, 2, 0).numpy())
    axes[4, 1].set_title('Predicted reconstructed diffuse')
    axes[4, 2].imshow(diffuse_error.numpy(), cmap='magma')
    axes[4, 2].set_title('Diffuse reconstruction error')

    for ax in axes.ravel():
        ax.axis('off')

    fig.suptitle(f'Fixed validation inference at epoch {epoch} ({train_type}-parameterization)', fontsize=16)
    plt.tight_layout()
    plt.show()


def run_eval(
    model: torch.nn.Module,
    loader,
    diffuser: diff.Diffuser,
    fixed_condition: torch.Tensor,
    fixed_target: torch.Tensor,
    fixed_diffuse: torch.Tensor,
    fixed_noise: torch.Tensor,
    device: torch.device,
    epoch: int,
    train_type: str = 'e',
    max_batches: int | None = None,
):
    model.eval()
    val_noise_loss = evaluate_noise_loss(model, loader, diffuser, device, train_type=train_type, max_batches=max_batches)

    with torch.no_grad():
        prediction = diffuser.sample_from_noise(
            model,
            fixed_condition,
            parameterization=train_type,
            show_progress=False,
            initial_noise=fixed_noise,
        )

    target = fixed_target.squeeze(0)
    prediction = prediction.squeeze(0)
    true_diffuse = fixed_diffuse.squeeze(0)
    condition = fixed_condition.squeeze(0)
    predicted_diffuse = condition - prediction

    sample_mse = float(F.mse_loss(prediction, target).item())
    sample_mae = float((prediction - target).abs().mean().item())
    reconstructed_diffuse_mse = float(F.mse_loss(predicted_diffuse, true_diffuse).item())
    reconstructed_diffuse_mae = float((predicted_diffuse - true_diffuse).abs().mean().item())

    plot_prediction(target, prediction, condition, true_diffuse, epoch=epoch, train_type=train_type)

    model.train()
    return {
        'epoch': epoch,
        'val_noise_loss': val_noise_loss,
        'sample_mse': sample_mse,
        'sample_mae': sample_mae,
        'reconstructed_diffuse_mse': reconstructed_diffuse_mse,
        'reconstructed_diffuse_mae': reconstructed_diffuse_mae,
    }

In [ ]:
progress_bar = tqdm(total=EPOCHS * len(train_loader), desc=f'Training [{TRAIN_TYPE}]', dynamic_ncols=True)
train_loss_history = []
eval_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = train_step(model, batch, diffuser, DEVICE, train_type=TRAIN_TYPE)
        loss.backward()
        optimizer.step()
        update_ema(ema_model, model, decay=EMA_DECAY)

        epoch_loss += float(loss.item())
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    epoch_loss /= len(train_loader)
    train_loss_history.append(epoch_loss)
    scheduler.step()

    print(f'Epoch {epoch:4d} | train loss {epoch_loss:.6f}')

    if epoch % CHECKPOINT_EVERY == 0:
        checkpoint_save(model, optimizer, epoch_loss, epoch, TRAINER_SAVE_PATH, TRAIN_TYPE)

    if epoch % EVAL_EVERY == 0:
        metrics = run_eval(
            ema_model,
            val_loader,
            diffuser,
            fixed_val_condition,
            fixed_val_target,
            fixed_val_diffuse,
            fixed_eval_noise,
            DEVICE,
            epoch,
            train_type=TRAIN_TYPE,
            max_batches=VAL_BATCHES,
        )
        eval_history.append(metrics)
        print(
            f"Eval {epoch:4d} | val objective {metrics['val_noise_loss']:.6f} | "
            f"diff mse {metrics['sample_mse']:.6f} | recon diffuse mse {metrics['reconstructed_diffuse_mse']:.6f}"
        )

progress_bar.close()

torch.save(
    {
        'epoch': EPOCHS,
        'train_type': TRAIN_TYPE,
        'model_state_dict': model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss_history': train_loss_history,
        'eval_history': eval_history,
    },
    FINAL_MODEL_PATH,
)
print(f'Final model saved to {FINAL_MODEL_PATH}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(train_loss_history) + 1), train_loss_history, label='train loss')
axes[0].set_title(f'Train Loss ({TRAIN_TYPE})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)
axes[0].legend()

if eval_history:
    eval_epochs = [item['epoch'] for item in eval_history]
    axes[1].plot(eval_epochs, [item['val_noise_loss'] for item in eval_history], label='val objective')
    axes[1].plot(eval_epochs, [item['reconstructed_diffuse_mse'] for item in eval_history], label='recon diffuse mse')
    axes[1].set_title(f'Validation Curves ({TRAIN_TYPE})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric')
    axes[1].grid(True)
    axes[1].legend()
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
TEST_GLOSSY_DIR = PSD_ROOT / 'PSD_Test' / 'PSD_Test_specular'
TEST_DIFFUSE_DIR = PSD_ROOT / 'PSD_Test' / 'PSD_Test_diffuse'
TEST_BATCH_SIZE = 1
TEST_EVAL_SEED = 123

try:
    from skimage.metrics import structural_similarity as structural_similarity
except ImportError:
    structural_similarity = None


class PSDTestMetricsDataset(Dataset):
    def __init__(self, glossy_dir: Path, diffuse_dir: Path, image_size: int):
        self.glossy_dir = Path(glossy_dir)
        self.diffuse_dir = Path(diffuse_dir)
        glossy_files = {path.name: path for path in self.glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS}
        diffuse_files = {path.name: path for path in self.diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS}
        self.names = sorted(set(glossy_files) & set(diffuse_files))
        if not self.names:
            raise RuntimeError(f'No PSD test pairs found in {self.glossy_dir} and {self.diffuse_dir}')
        self.glossy_files = glossy_files
        self.diffuse_files = diffuse_files
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, idx: int):
        name = self.names[idx]
        glossy = load_rgb_tensor(self.glossy_files[name], self.image_size)
        diffuse = load_rgb_tensor(self.diffuse_files[name], self.image_size)
        return glossy, diffuse, name


def compute_batch_psnr(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    mse = F.mse_loss(prediction, target, reduction='none').flatten(1).mean(dim=1)
    return 10.0 * torch.log10(1.0 / mse.clamp_min(1e-10))


def compute_batch_ssim(prediction: torch.Tensor, target: torch.Tensor) -> list[float]:
    if structural_similarity is None:
        return []
    scores = []
    for pred_img, true_img in zip(prediction, target):
        pred_np = pred_img.detach().cpu().permute(1, 2, 0).numpy()
        true_np = true_img.detach().cpu().permute(1, 2, 0).numpy()
        scores.append(float(structural_similarity(true_np, pred_np, channel_axis=-1, data_range=1.0)))
    return scores


if not TEST_GLOSSY_DIR.exists() or not TEST_DIFFUSE_DIR.exists():
    raise FileNotFoundError(f'Missing PSD test directories: {TEST_GLOSSY_DIR} and/or {TEST_DIFFUSE_DIR}')

test_dataset = PSDTestMetricsDataset(TEST_GLOSSY_DIR, TEST_DIFFUSE_DIR, image_size=IMAGE_SIZE)
test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)
eval_model = ema_model if 'ema_model' in globals() else model
predicts_difference = 'fixed_val_diffuse' in globals()

torch.manual_seed(TEST_EVAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TEST_EVAL_SEED)
test_noise_generator = torch.Generator()
test_noise_generator.manual_seed(TEST_EVAL_SEED)

psnr_scores = []
ssim_scores = []
test_rows = []

eval_model.eval()
with torch.no_grad():
    for condition, true_diffuse, names in tqdm(test_loader, desc='PSD test metrics'):
        condition = condition.to(DEVICE)
        true_diffuse = true_diffuse.to(DEVICE)

        if 'ddpm_sample_from_noise_safe' in globals():
            raw_prediction = ddpm_sample_from_noise_safe(eval_model, diffuser, condition, show_progress=False)
        else:
            init_noise = torch.randn(true_diffuse.shape, generator=test_noise_generator, dtype=true_diffuse.dtype).to(DEVICE)
            raw_prediction = diffuser.sample_from_noise(
                eval_model,
                condition,
                parameterization=TRAIN_TYPE,
                show_progress=False,
                initial_noise=init_noise,
            )

        predicted_diffuse = (condition - raw_prediction).clamp(0.0, 1.0) if predicts_difference else raw_prediction.clamp(0.0, 1.0)
        batch_psnr = compute_batch_psnr(predicted_diffuse, true_diffuse)
        batch_ssim = compute_batch_ssim(predicted_diffuse, true_diffuse)

        for idx, name in enumerate(names):
            row = {'name': name, 'psnr': float(batch_psnr[idx].item())}
            if batch_ssim:
                row['ssim'] = float(batch_ssim[idx])
            test_rows.append(row)
            psnr_scores.append(row['psnr'])
            if 'ssim' in row:
                ssim_scores.append(row['ssim'])

psd_test_metrics = {
    'num_samples': len(test_dataset),
    'predicts_difference': predicts_difference,
    'train_type': TRAIN_TYPE,
    'mean_psnr': float(np.mean(psnr_scores)),
    'median_psnr': float(np.median(psnr_scores)),
    'min_psnr': float(np.min(psnr_scores)),
    'max_psnr': float(np.max(psnr_scores)),
    'mean_ssim': float(np.mean(ssim_scores)) if ssim_scores else None,
    'per_sample': test_rows,
}

print(f"PSD test samples: {psd_test_metrics['num_samples']}")
print(f"Predicts difference: {psd_test_metrics['predicts_difference']}")
print(f"Train type: {psd_test_metrics['train_type']}")
print(f"Mean PSNR: {psd_test_metrics['mean_psnr']:.4f} dB")
print(f"Median PSNR: {psd_test_metrics['median_psnr']:.4f} dB")
if psd_test_metrics['mean_ssim'] is None:
    print('Mean SSIM: skipped because scikit-image is not installed')
else:
    print(f"Mean SSIM: {psd_test_metrics['mean_ssim']:.6f}")

sorted(psd_test_metrics['per_sample'], key=lambda item: item['psnr'])[:5]
